# 3SAT with TNReason

This notebook demonstrates solving Boolean 3-SAT problems using the original `tnreason` library.

It covers:
- Parsing DIMACS CNF format
- Converting CNF to tnreason's nested-list formula language
- Building a `HybridKnowledgeBase`
- Checking satisfiability and (optionally) estimating model count
- Querying marginals and extracting a satisfying assignment
- Generating 3SAT test data and check for its vadility

Requirements: `tnreason` vendored in `tnreason/` folder of this repo.


In [ ]:
# Imports and environment setup
# - json: pretty printing of results
# - typing: lightweight type hints in helper functions
# - tnreason modules:
#   * application.distributions.HybridKnowledgeBase: holds formulas (hard/weighted)
#   * application.deductive.InferenceProvider: runs queries over the TN
#   * application.cnf_to_cores: optional CNF-based encoders (alternative pipeline)
import json
from typing import List, Tuple

import sys, importlib.util
print(sys.version)
print(sys.executable)
print(importlib.util.find_spec("tnreason"))

# Ensure repo root is on sys.path so `import tnreason` works when running from subfolders
import sys
from pathlib import Path
try:
    base_dir = Path(__file__).resolve().parent
except NameError:
    base_dir = Path.cwd()

repo_root = None
for candidate in [base_dir, *base_dir.parents]:
    if (candidate / 'tnreason').is_dir():
        repo_root = candidate
        break
if repo_root is not None and str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from tnreason.application.distributions import HybridKnowledgeBase
from tnreason.application.deductive import InferenceProvider
from tnreason.application.cnf_to_cores import formula_to_sparseCore, cnf_to_dict

print("TNReason imported.")


ModuleNotFoundError: No module named 'sortedcontainers'

In [ ]:
# DIMACS parsing and conversion to tnreason nested-list
# This cell provides two utilities:
# 1) parse_dimacs: read a DIMACS CNF string into (num_vars, list of integer clauses)
#    - Lines starting with c/p/%/s are ignored (comments/headers/terminators)
#    - Clause lines are sequences of integers terminated by 0
# 2) dimacs_to_nested_list: turn the parsed clauses into tnreason's nested-list logic format
#    - The nested-list language uses strings and small lists: ["and", ...], ["or", ...], ["not", "x1"], "x3", etc.


def parse_dimacs(dimacs_str: str) -> Tuple[int, List[List[int]]]:
    """Parse DIMACS CNF string into (num_vars, clauses), robust to trailing comments like '%'."""
    clauses: List[List[int]] = []
    num_vars = None
    for raw in dimacs_str.splitlines():
        line = raw.strip()
        if not line:
            continue
        if line.startswith('c') or line.startswith('p') or line.startswith('%') or line.startswith('s'):
            if line.startswith('p'):
                parts = line.split()
                if len(parts) >= 4 and parts[1] == 'cnf':
                    try:
                        num_vars = int(parts[2])
                    except Exception:
                        num_vars = None
            continue
        tokens = line.split()
        cur: List[int] = []
        for tok in tokens:
            if tok == '0':
                break
            if tok.startswith('%'):
                break
            try:
                cur.append(int(tok))
            except ValueError:
                # skip non-integer tokens in clause lines
                continue
        if cur:
            clauses.append(cur)
    if num_vars is None:
        num_vars = max(abs(l) for cl in clauses for l in cl) if clauses else 0
    return num_vars, clauses


def dimacs_to_nested_list(num_vars: int, clauses: List[List[int]], var_prefix: str = "x"):
    """Convert DIMACS to tnreason nested list expression: ["and", ["or", ...], ...]."""
    and_args = []
    for cl in clauses:
        or_args = []
        for lit in cl:
            v = f"{var_prefix}{abs(lit)}"
            if lit < 0:
                or_args.append(["not", v])
            else:
                or_args.append(v)
        # Each clause becomes an OR node with its three literals
        or_node = ["or"] + or_args
        and_args.append(or_node)
    # The whole CNF becomes an AND over all clause ORs
    return ["and"] + and_args if and_args else ["and"]

print("DIMACS helpers ready.")


DIMACS helpers ready.


In [ ]:
# Normalize formula: binarize n-ary AND/OR to binary trees
# tnreason connectives are binary (AND/OR take exactly two inputs).
# This helper rewrites ["or", a, b, c] into ["or", ["or", a, b], c] (and similarly for AND),
# recursively through the expression.

def binarize_connectives(expr):
    """Convert any n-ary [op, a, b, c, ...] into a binary tree: [op, [op, a, b], c], ...
    Applies recursively to sub-expressions.
    """
    if isinstance(expr, str):
        return expr
    if len(expr) == 1:
        return [binarize_connectives(expr[0])]
    if len(expr) == 2:
        op, a = expr[0], expr[1]
        return [op, binarize_connectives(a)]
    # len >= 3
    op = expr[0]
    # start with first two
    acc = [op, binarize_connectives(expr[1]), binarize_connectives(expr[2])]
    for i in range(3, len(expr)):
        acc = [op, acc, binarize_connectives(expr[i])]
    return acc

print("Binarization helper ready.")


Binarization helper ready.


In [ ]:
# Build KB and compute satisfiability / model count
# We store the CNF as a single hard fact "F" in a HybridKnowledgeBase.
# tnreason will build a tensor network for the logic, and contracting it tells us
# whether there exists any satisfying assignment (and how many).

def build_kb_from_formula(formula):
    """Create a HybridKnowledgeBase where `formula` is a hard fact."""
    return HybridKnowledgeBase(facts={"F": formula})


def is_satisfiable(kb: HybridKnowledgeBase) -> bool:
    """Use tnreason hard-cores contraction to decide satisfiability."""
    return kb.is_satisfiable()


def estimate_model_count_via_partition(kb: HybridKnowledgeBase) -> float:
    """Estimate model count as partition with only hard constraints (Boolean vars) when feasible.
    For strict 3SAT with only facts and 2-valued atoms, this equals the number of satisfying assignments.
    """
    # For KBs without weights, partition equals count of satisfying assignments if dimensions are all 2.
    # Build cores for hard constraints only and contract with no open colors.
    try:
        from tnreason import engine
    except ImportError:
        import sys
        from pathlib import Path
        try:
            base_dir = Path(__file__).resolve().parent
        except NameError:
            base_dir = Path.cwd()
        for candidate in [base_dir, *base_dir.parents]:
            if (candidate / 'tnreason').is_dir():
                if str(candidate) not in sys.path:
                    sys.path.insert(0, str(candidate))
                break
        from tnreason import engine
    hard_cores = kb.create_caNetwork(hardOnly=True).create_cores()
    contracted = engine.contract(hard_cores, openColors=[])
    return float(contracted[:])

print("KB helpers ready.")


KB helpers ready.


In [ ]:
# Inference: marginals and a satisfying assignment
# - compute_marginals: P(xi=True | F) for each atom xi by contracting with xi open and normalizing
# - find_satisfying_assignment: try exact MAP over all atoms; fallback to greedy argmax of marginals

def compute_marginals(kb: HybridKnowledgeBase):
    """Return per-atom marginal P(atom=True | F)."""
    inferer = InferenceProvider(kb)
    # Identify atoms from KB (colors with suffix suf.disVarSuf)
    from tnreason.application import script_transform as st
    from tnreason.representation import suffixes as suf
    atom_colors = st.get_all_atom_colors({"F": kb.facts["F"]})
    # Build marginal dict keyed by plain variable names ("x1", ...)
    marginals = {}
    for atom_color in sorted(atom_colors):
        # atom_color like "x1" + suf.disVarSuf; query expects atom expression string "x1"
        atom_str = atom_color[:-len(suf.disVarSuf)]
        p_true = inferer.ask(atom_str)
        marginals[atom_str] = float(p_true)
    return marginals


def find_satisfying_assignment(kb: HybridKnowledgeBase):
    """Attempt to retrieve an exact MAP assignment over atoms. Falls back to greedy marginals if exact MAP not available."""
    inferer = InferenceProvider(kb)
    from tnreason.application import script_transform as st
    from tnreason.representation import suffixes as suf
    atom_colors = st.get_all_atom_colors({"F": kb.facts["F"]})
    try:
        # exact_map_query returns dict from colors to argmax slice value
        argmax = inferer.exact_map_query(atom_colors)
        # Convert color->int to var->bool
        assignment = {c[:-len(suf.disVarSuf)]: bool(v) for c, v in argmax.items()}
        return assignment
    except Exception:
        # Fallback: greedy from marginals
        m = compute_marginals(kb)
        return {k: (v >= 0.5) for k, v in m.items()}

print("Inference helpers ready.")


Inference helpers ready.


## Example: Tiny 3-CNF

We parse a small DIMACS CNF, convert it to tnreason's formula, build the KB, check satisfiability, estimate the model count, and derive a satisfying assignment.


In [ ]:
# Clean ASCII demo and binarized formula
s = """
 p cnf 4 3
 1 2 3 0
 -1 2 4 0
 2 3 4 0
 """

nvars, clauses = parse_dimacs(s)
formula = dimacs_to_nested_list(nvars, clauses)
formula = binarize_connectives(formula)
print("Formula (binarized):\n", json.dumps(formula, indent=2))
print("___"*60)

kb = build_kb_from_formula(formula)
print("KB satisfiable?", is_satisfiable(kb))

Z = estimate_model_count_via_partition(kb)
print("Model count (exact, hard constraints):", Z)

marginals = compute_marginals(kb)
print("Marginals P(xi=True|F):", marginals)

assignment = find_satisfying_assignment(kb)
print("Satisfying assignment (MAP or greedy):", assignment)


Formula (binarized):
 [
  "and",
  [
    "and",
    [
      "or",
      [
        "or",
        "x1",
        "x2"
      ],
      "x3"
    ],
    [
      "or",
      [
        "or",
        [
          "not",
          "x1"
        ],
        "x2"
      ],
      "x4"
    ]
  ],
  [
    "or",
    [
      "or",
      "x2",
      "x3"
    ],
    "x4"
  ]
]
____________________________________________________________________________________________________________________________________________________________________________________
KB satisfiable? True
Model count (exact, hard constraints): 12.0
Marginals P(xi=True|F): {'x1': 0.5, 'x2': 0.6666666666666666, 'x3': 0.5833333333333334, 'x4': 0.5833333333333334}
Satisfying assignment (MAP or greedy): {'x3': False, 'x1': False, 'x2': True, 'x4': False}


In [ ]:
# Dataset utilities: UF20-91 tar.gz loader
# Helpers to list and extract DIMACS CNFs from the provided tarball so we can
# feed them into the tnreason pipeline.
import tarfile, random
from pathlib import Path

DATASET_PATH = Path("uf20-91.tar.gz")


def list_cnf_members(tar_path: Path):
    """Return tar members that are .cnf files."""
    with tarfile.open(tar_path, "r:gz") as tar:
        return [m for m in tar.getmembers() if m.isfile() and m.name.endswith(".cnf")]


def read_member_text(tar, member):
    f = tar.extractfile(member)
    return f.read().decode("utf-8", errors="ignore")


def load_random_cnf_from_tar(tar_path: Path, seed: 'int | None' = None):
    rng = random.Random(seed)
    with tarfile.open(tar_path, "r:gz") as tar:
        cnf_members = [m for m in tar.getmembers() if m.isfile() and m.name.endswith(".cnf")]
        if not cnf_members:
            raise FileNotFoundError("No .cnf files in archive")
        member = rng.choice(cnf_members)
        text = read_member_text(tar, member)
        return member.name, text


def load_cnf_by_index(tar_path: Path, index: int):
    with tarfile.open(tar_path, "r:gz") as tar:
        cnf_members = [m for m in tar.getmembers() if m.isfile() and m.name.endswith(".cnf")]
        if not (0 <= index < len(cnf_members)):
            raise IndexError(f"index {index} out of range (0..{len(cnf_members)-1})")
        member = cnf_members[index]
        return member.name, read_member_text(tar, member)

print("Dataset utilities ready.")


Dataset utilities ready.


Problem: einstein summation only works with up to 62 unique labels. For 3SAT standard test set we need 91.

In [ ]:
# Random 3-SAT generator (DIMACS) with optional guaranteed satisfiability
import random
from typing import List, Tuple, Optional

def generate_random_3sat_dimacs(
    n_vars: int = 10,
    n_clauses: int = 30,
    ensure_sat: bool = True,
    seed: Optional[int] = None,
    forbid_tautologies: bool = True,
    forbid_duplicate_clauses: bool = True,
) -> Tuple[str, Optional[List[int]]]:
    """
    Create a random 3-SAT instance in DIMACS CNF format.
    If ensure_sat=True, also returns a satisfying assignment (list of 0/1 of length n_vars).

    Returns:
      dimacs_str, witness (witness is None if ensure_sat=False)
    """
    # Initialize a local RNG so function is deterministic for a given seed
    rng = random.Random(seed)

    # Optional hidden satisfying assignment. If ensure_sat=True we bias clause generation
    # so that each clause is satisfied by this witness (at least one literal matches it).
    witness = None
    if ensure_sat:
        # Witness is a list of 0/1 values, 1-based DIMACS variable k maps to witness[k-1]
        witness = [rng.randint(0, 1) for _ in range(n_vars)]

    # Keep track of generated clauses and a normalized set to detect duplicates
    clauses_set = set()              # holds tuples for deduplication
    clauses: List[List[int]] = []    # actual list of 3-literal integer clauses

    def lit_sign_for_var(var_idx: int) -> int:
        """Choose literal sign (+ for x, - for ¬x) for variable with 0-based index var_idx.
        If ensure_sat, we bias signs so the clause tends to have at least one literal matching the witness.
        """
        if not ensure_sat:
            # Unbiased sign if we don't need to guarantee satisfiability
            return 1 if rng.random() < 0.5 else -1
        # With ensure_sat=True, favor the sign that matches the witness around 50% of the time
        target = witness[var_idx]
        if rng.random() < 0.5:
            # Match witness directly (x if witness=1, ¬x if witness=0)
            return +1 if target == 1 else -1
        else:
            # Otherwise pick a random sign
            return 1 if rng.random() < 0.5 else -1

    # Generate clauses until we have the requested count
    while len(clauses) < n_clauses:
        # Pick 3 distinct variable indices (0-based) for a 3-literal clause
        vars3 = rng.sample(range(n_vars), 3)

        # Build the 3 literals (DIMACS: positive k means x_k, negative -k means ¬x_k)
        lits = []
        for vi in vars3:
            sgn = lit_sign_for_var(vi)
            lit = (vi + 1) * sgn   # convert 0-based to 1-based variable id
            lits.append(lit)

        # Optionally forbid tautologies: clauses containing both x and ¬x of the same variable.
        # Detect by checking if absolute var ids collapse to < 3 distinct entries.
        if forbid_tautologies:
            absset = set(abs(x) for x in lits)
            if len(absset) < 3:
                # Resample a new clause
                continue

        # If we must ensure satisfiable, enforce that at least one literal is satisfied by the witness.
        if ensure_sat:
            def sat_by_witness(l):
                # A literal l is satisfied by witness iff:
                #  - l > 0 and witness[v] == 1, or
                #  - l < 0 and witness[v] == 0, where v = abs(l) - 1
                v = abs(l) - 1
                val = witness[v]
                return (l > 0 and val == 1) or (l < 0 and val == 0)

            if not any(sat_by_witness(l) for l in lits):
                # If none match, flip one randomly to match the witness so the clause is satisfied
                idx = rng.randrange(3)
                v = abs(lits[idx]) - 1
                lits[idx] = (v + 1) if witness[v] == 1 else -(v + 1)

        # Normalize clause for deduplication: sort by (abs(var), signed value) to get a canonical tuple
        norm = tuple(sorted(lits, key=lambda z: (abs(z), z)))
        if forbid_duplicate_clauses and norm in clauses_set:
            # Skip duplicates to keep the instance diverse
            continue
        clauses_set.add(norm)
        clauses.append(lits)

    # Build the DIMACS string:
    #  - header: p cnf <n_vars> <n_clauses>
    #  - each clause as three literals followed by 0 terminator
    lines = [f"p cnf {n_vars} {n_clauses}"]
    for cl in clauses:
        lines.append(f"{cl[0]} {cl[1]} {cl[2]} 0")
    dimacs = "\n".join(lines) + "\n"

    # Return the DIMACS text and the hidden witness (or None if not requested)
    return dimacs, witness

print("3-SAT generator ready.")


3-SAT generator ready.


In [ ]:
# Ground-truth SAT utilities: pycosat if available, else brute-force for small n
import itertools

try:
    import pycosat  # pip install pycosat
    HAS_PYCOSAT = True
except Exception:
    HAS_PYCOSAT = False


def dimacs_to_clause_list(dimacs_str: str):
    """Return List[List[int]] of clauses, ignoring comments/headers."""
    _, clauses = parse_dimacs(dimacs_str)
    return clauses


def brute_force_solutions(n_vars: int, clauses: List[List[int]]):
    """Return list of satisfying 0/1 assignments of length n_vars. For small n only (expensive)."""
    sols = []
    for bits in itertools.product([0,1], repeat=n_vars):
        ok = True
        for cl in clauses:
            sat = False
            for l in cl:
                v = abs(l)-1
                val = bits[v]
                if (l > 0 and val == 1) or (l < 0 and val == 0):
                    sat = True
                    break
            if not sat:
                ok = False
                break
        if ok:
            sols.append(list(bits))
    return sols


def ground_truth_solve(dimacs_str: str):
    n_vars, clauses = parse_dimacs(dimacs_str)
    if HAS_PYCOSAT:
        sols = []
        for sol in pycosat.itersolve(clauses):
            # pycosat returns list of ints with signs; convert to 0/1 assignment
            assign = [0]*n_vars
            for lit in sol:
                v = abs(lit)-1
                if lit > 0:
                    assign[v] = 1
                else:
                    assign[v] = 0
            sols.append(assign)
    else:
        # fallback: only viable for small n
        sols = brute_force_solutions(n_vars, clauses)
    # marginals
    count = len(sols)
    if count == 0:
        marg = [0.0]*n_vars
    else:
        marg = [sum(sol[i] for sol in sols)/count for i in range(n_vars)]
    return dict(
        satisfiable=(count>0),
        count=count,
        marginals=marg,
        n_vars=n_vars,
        n_clauses=len(clauses),
    )

print("Ground-truth SAT utilities ready.")


Ground-truth SAT utilities ready.


In [ ]:
# Runner: generate CNF, get ground truth, run TNReason, compare
try:
    from tnreason import engine
except ImportError:
    import sys
    from pathlib import Path
    try:
        base_dir = Path(__file__).resolve().parent
    except NameError:
        base_dir = Path.cwd()
    for candidate in [base_dir, *base_dir.parents]:
        if (candidate / 'tnreason').is_dir():
            if str(candidate) not in sys.path:
                sys.path.insert(0, str(candidate))
            break
    from tnreason import engine

# engine.defaultContractionMethod = "CorewiseContractor"  # safer on larger graphs


def run_trial(n_vars=10, n_clauses=30, ensure_sat=True, seed=None):
    dimacs, wit = generate_random_3sat_dimacs(n_vars=n_vars, n_clauses=n_clauses, ensure_sat=ensure_sat, seed=seed)
    gt = ground_truth_solve(dimacs)

    # Build TNReason KB
    nvars, clauses = parse_dimacs(dimacs)
    formula = dimacs_to_nested_list(nvars, clauses)
    formula = binarize_connectives(formula)
    kb = build_kb_from_formula(formula)

    # Satisfiable and count (count may be large; einsum might hit symbol limit)
    try:
        sat_tn = bool(is_satisfiable(kb))
    except Exception as e:
        sat_tn = None
        print("TNReason satisfiable? error:", repr(e))
    try:
        Z_tn = float(estimate_model_count_via_partition(kb))
    except Exception as e:
        Z_tn = None
        print("TNReason model count error:", repr(e))

    # Marginals
    try:
        marg_tn = {k: float(v) for k, v in compute_marginals(kb).items()}
    except Exception as e:
        marg_tn = None
        print("TNReason marginals error:", repr(e))

    # Compare (only for parts that are available)
    report = {
        "n_vars": nvars,
        "n_clauses": len(clauses),
        "gt_satisfiable": gt["satisfiable"],
        "tn_satisfiable": sat_tn,
        "gt_count": gt["count"],
        "tn_count": Z_tn,
    }
    if marg_tn is not None:
        # map tn marginals dict {"x1":p,...} to list order
        tn_marg_list = [marg_tn.get(f"x{i+1}") for i in range(nvars)]
        report["tn_marginals"] = tn_marg_list
        report["gt_marginals"] = gt["marginals"]
    return report

# Example run (small to allow brute-force if pycosat not installed)
res = run_trial(n_vars=7, n_clauses=7, ensure_sat=True, seed=123)
print("Trial report:\n", json.dumps(res, indent=2))


TNReason satisfiable? error: ValueError('too many subscripts in einsum')
TNReason model count error: ValueError('too many subscripts in einsum')
TNReason marginals error: ValueError('too many subscripts in einsum')
Trial report:
 {
  "n_vars": 7,
  "n_clauses": 7,
  "gt_satisfiable": true,
  "tn_satisfiable": null,
  "gt_count": 54,
  "tn_count": null
}


In [ ]:
# Run a random UF20-91 instance through tnreason
try:
    from tnreason import engine
except ImportError:
    import sys
    from pathlib import Path
    try:
        base_dir = Path(__file__).resolve().parent
    except NameError:
        base_dir = Path.cwd()
    for candidate in [base_dir, *base_dir.parents]:
        if (candidate / 'tnreason').is_dir():
            if str(candidate) not in sys.path:
                sys.path.insert(0, str(candidate))
            break
    from tnreason import engine

if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Dataset not found at {DATASET_PATH}")

name, cnf_text = load_random_cnf_from_tar(DATASET_PATH, seed=None)
print(f"Picked: {name}")

nvars, clauses = parse_dimacs(cnf_text)
formula = dimacs_to_nested_list(nvars, clauses, var_prefix="x")
formula = binarize_connectives(formula)
print("nvars:", nvars, "num_clauses:", len(clauses))

kb = build_kb_from_formula(formula)
print("KB satisfiable?", is_satisfiable(kb))

# For speed/robustness on larger instances, you can switch the contractor if needed:
# engine.defaultContractionMethod = "CorewiseContractor"

Z = estimate_model_count_via_partition(kb)
print("Model count (exact, hard constraints):", Z)

# Optionally, show a small subset of marginals (variables 1..10)
all_m = compute_marginals(kb)
subset = {k: all_m[k] for k in sorted(all_m, key=lambda s: int(s[1:]))[:10]}
print("Marginals (first 10 vars):", subset)

assignment = find_satisfying_assignment(kb)
print("Satisfying assignment (MAP or greedy):", {k: assignment[k] for k in sorted(assignment)[:10]}, "...")


Picked: uf20-034.cnf
nvars: 20 num_clauses: 91


ValueError: Length of Contraction is too large for Einsum!